## Model Profiling
Profiling the models using the model.profile function by providing the compiled .tflite or .zip model path and automatically uploading profiling artifacts and history logs to Databricks Unity Catalog Volumes path specified.

(Note:- We are installing whole Silicon Labs MlOps SDK for running the notebook)

In [ ]:
dbutils.library.restartPython()

In [0]:
%pip install "/Workspace/Users/mail@domain.com/silabs-mlops-cli"

Processing ./silabs-mlops-cli
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 138.2 MB/s eta 0:00:00
  Created wheel for silabs-mlops: filename=silabs_mlops-0.1.0-py3-none-any.whl size=271518 sha256=39ef236712914ab628bccf82ddb5ea5fe58fb5d45065b1cd009d3ccf0fe25cbd
  Stored in directory: /home/spark-adae050b-86ea-4a7f-98be-72/.cache/pip/wheels/fa/25/8c/d1b72e9d15cb07512285869b82b6d18c8a6141b736f74b839f
Successfully built silabs-mlops
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-adae050

### Global Credentials Configuration

Call this ONLY if you have NOT already configured the global credentials. If you already called data.config() earlier (e.g., during data ingestion)you DO NOT need to call it again before profiling as both data and model modules reuse the same config.

In [0]:
from silabs_mlops import data
data.config(
    server_endpoint="<<SERVER_ENDPOINT_URL>>",
    workspace_url="<<WORKSPACE_URL>>",
    table_name="mlops_dev.default.stream_audio_events",
    client_id="<<CLIENT_ID>>",
    client_secret="<<CLIENT_SECRET>>"
)


[OK] Configuration saved. You can now use data.ingest() to send data.


In [0]:
from silabs_mlops import model
model_path = "/Workspace/Users/user_email.com/models/Custom_model.tflite" 
print("\n--- [A] Local Simulation & Cloud Upload ---")
try:
    result = model.profile(
        model_path=model_path,
        use_simulator=True,          # Set to True to run the model profiling locally using the simulator
        volume_path="/Volumes/mlops_dev/default/profiling_results",
        profiler_path="/Workspace/Users/user_email.com/mvp_profiler"
    )
    # The result object contains all the extracted data:
    print(f"  \u2713 Model:         {result.model_name}")
    print(f"  \u2713 Arena Size:    {result.arena_size_kb:.1f} KB")
    print(f"  \u2713 Total MACs:    {result.total_macs:,}")
    print(f"  \u2713 Remote Folder: {result.output_dir}")
    print(f"  \u2713 History Log:   {result.history_log_path}")
except Exception as e:
    # If there is a failure, the script will crash here
    # but the history.log will STILL upload to the volume path.
    print(f"  [!] Profiling failed -> {e}")


--- [A] Local Simulation & Cloud Upload ---
Using profiler command: /Workspace/Users/pajoshi@silabs.com/mvp_profiler
  Mode:      Simulator (Local)

  SILICON LABS NPU MODEL PROFILER
  Model:     Custom_model.tflite
  Device ID: None
  Output:    /tmp/npu_prof__gvqb6wq

Running: /Workspace/Users/pajoshi@silabs.com/mvp_profiler /Workspace/Users/pajoshi@silabs.com/models/Custom_model.tflite --accelerator mvpv1 --output /tmp/npu_prof__gvqb6wq

Logging profiling history to: /tmp/npu_prof__gvqb6wq/profiling_history.log

Compiling model ...
Profiling model on simulator
Generating /tmp/npu_prof__gvqb6wq/Custom_model-profiling_summary.txt
Generating /tmp/npu_prof__gvqb6wq/Custom_model-profiling_results.yaml
Generating /home/spark-adae050b-86ea-4a7f-98be-72/.npu_toolkit/compiled_models/Custom_model/20260324T053949/profiler/profiling_summary.txt
Generating /home/spark-adae050b-86ea-4a7f-98be-72/.npu_toolkit/compiled_models/Custom_model/20260324T053949/profiler/profiling_results.yaml
Profiling S

### MLflow Integration
Log profiling metrics, parameters, and artifacts to MLflow for tracking and comparing model profiling runs across different models and configurations. The profiling result object from the cell above is used to populate the MLflow run.

In [0]:
import mlflow
import json
import os

# ── Configure MLflow to use Unity Catalog ──
mlflow.set_registry_uri("databricks-uc")

# Experiment lives in workspace; artifacts are stored in the volume
experiment_name = "/Users/user_email.com/model_profiling_experiments"
volume_path = "/Volumes/mlops_dev/default/profiling_results"

try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=volume_path
    )
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_id=experiment_id)

# ── Parse the profiling report for detailed metrics ──
report_path = os.path.join(result.output_dir, "report.json")
detailed_metrics = {}
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        report = json.load(f)
    detailed_metrics = report

# ── Start MLflow Run ──
with mlflow.start_run(run_name=f"profile-{result.model_name}") as run:
    
    # Log Parameters
    mlflow.log_param("model_name", result.model_name)
    mlflow.log_param("model_path", model_path)
    mlflow.log_param("platform", "simulator")
    mlflow.log_param("accelerator", "mvpv1")
    mlflow.log_param("use_simulator", True)
    
    for key in ["input_shape", "output_shape", "input_dtype", "output_dtype", 
                "accelerator_variant", "npu_toolkit_version"]:
        if key in detailed_metrics:
            mlflow.log_param(key, detailed_metrics[key])
    
    # Log Profiling Metrics
    mlflow.log_metric("arena_size_kb", result.arena_size_kb)
    mlflow.log_metric("total_macs", result.total_macs)
    
    metric_keys = {
        "flash_bytes": "flash_bytes",
        "sram_bytes": "sram_bytes",
        "op_count": "op_count",
        "layer_count": "layer_count",
        "unsupported_layer_count": "unsupported_layer_count",
        "accelerator_cycles": "accelerator_cycles",
        "mac_per_cycle": "mac_per_cycle",
    }
    for mlflow_key, report_key in metric_keys.items():
        if report_key in detailed_metrics:
            try:
                mlflow.log_metric(mlflow_key, float(detailed_metrics[report_key]))
            except (ValueError, TypeError):
                pass
    
    # Log Artifacts (stored in the volume path)
    if os.path.exists(model_path):
        mlflow.log_artifact(model_path, artifact_path="model")
    
    if os.path.isdir(result.output_dir):
        mlflow.log_artifacts(result.output_dir, artifact_path="profiling_output")
    
    # Tags for easy filtering
    mlflow.set_tag("task", "model_profiling")
    mlflow.set_tag("model_type", "tflite")
    mlflow.set_tag("accelerator", "mvpv1")
    
    print(f"✓ MLflow Run ID:     {run.info.run_id}")
    print(f"✓ Experiment:        {experiment_name}")
    print(f"✓ Artifact Location: {volume_path}")
    print(f"✓ Metrics logged:    arena_size_kb={result.arena_size_kb}, total_macs={result.total_macs:,}")
    print(f"✓ Artifacts logged:  model .tflite + profiling outputs")
    print(f"\n  View run: {mlflow.get_tracking_uri()}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

✓ MLflow Run ID:     19cc00ca4f0640fda6a4679a66d76be5
✓ Experiment:        /Users/pajoshi@silabs.com/model_profiling_experiments
✓ Artifact Location: /Volumes/mlops_dev/default/profiling_results
✓ Metrics logged:    arena_size_kb=91.7, total_macs=8,100,000
✓ Artifacts logged:  model .tflite + profiling outputs

  View run: databricks/#/experiments/888508805576752/runs/19cc00ca4f0640fda6a4679a66d76be5


### Upload To Model Registry
Once your seleted model is successfully profiled and validated, run the below scripts (cells 10-12) to upload the model to the model registry.

In [0]:
def register_single_file(file_path, uc_model_name, run_name):
    import mlflow, os
    from mlflow import MlflowClient
    import numpy as np
    from mlflow.models import infer_signature
    from mlflow.pyfunc import PythonModel

    print(f"[UC] Registering: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # Ensure no active run
    if mlflow.active_run():
        mlflow.end_run()

    # ---- Minimal pyfunc so UC accepts model directory ----
    class Identity(PythonModel):
        def predict(self, context, model_input):
            return model_input

    # Dummy signature (UC requires model signature)
    dummy_input  = np.zeros((1,1), dtype="float32")
    dummy_output = np.zeros((1,1), dtype="float32")
    signature    = infer_signature(dummy_input, dummy_output)

    # ---- START RUN ----
    with mlflow.start_run(run_name=run_name) as run:
        # Log as a pyfunc model with the file as an extra artifact
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=Identity(),
            signature=signature,
            input_example=dummy_input,
            artifacts={"model_file": file_path},
        )

        # Register the model
        mv = mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=uc_model_name,
        )

    print(f"[UC] Registered {uc_model_name} \u2192 version {mv.version}")
    return mv.version

In [0]:
teacher_v = register_single_file(
    "/Workspace/Users/user_email.com/models/Custom_model.teacher.h5",
    "mlops_dev.default.Custom_model_teacher",
    "teacher_register"
)

[UC] Registering: /Workspace/Users/pajoshi@silabs.com/models/Custom_model.teacher.h5


/local_disk0/.ephemeral_nfs/envs/pythonEnv-adae050b-86ea-4a7f-98be-72aef43d948f/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/03/24 05:51:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405615984054316.16.azuredatabricks.net/ml/experiments/888508805576752/models/m-d0c3b0ec4f3e47d4bb5bd98236239b74?o=7405615984054316
2026/03/24 05:51:12 INFO mlflow.pyfunc: Validating input example against model signature
Registered model 'mlops_dev.default.Custom_model_teacher' already exists. Creating a new version of this model...
2026/03/24 05:51:21 WARNING mlflow.tracking._model_registry.fluent: Run with id c32d25b7d2c5404a9883eb6b1e85

[UC] Registered mlops_dev.default.Custom_model_teacher → version 9


In [0]:
student_v = register_single_file(
    "/Workspace/Users/user_email.com/models/Custom_model.tflite",
    "mlops_dev.default.Custom_model_student",
    "student_register"
)

[UC] Registering: /Workspace/Users/pajoshi@silabs.com/models/Custom_model.tflite


/local_disk0/.ephemeral_nfs/envs/pythonEnv-adae050b-86ea-4a7f-98be-72aef43d948f/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/03/24 05:51:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405615984054316.16.azuredatabricks.net/ml/experiments/888508805576752/models/m-6e3b9ec9225b48c486001f115b8aa8ac?o=7405615984054316
2026/03/24 05:51:32 INFO mlflow.pyfunc: Validating input example against model signature
Registered model 'mlops_dev.default.Custom_model_student' already exists. Creating a new version of this model...
2026/03/24 05:51:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 692b496abf894d12a209422f7720

[UC] Registered mlops_dev.default.Custom_model_student → version 9


#### View Logs
All the actions performed on the library will be stored locally in json format. To view them in a structured format import the logger class and view each log event type (eg. Profiling, deployment, Data Ingestion)

In [0]:
from silabs_mlops.logs import Logger
logger = Logger()
#logger.view()
logger.view(event_type='profiling')


--- Local Log History (profiling) ---
TIMESTAMP            | TYPE             | LEVEL    | SOURCE          | MESSAGE
----------------------------------------------------------------------------------------------------
2026-03-24 05:38:38  | Profiling        | Info     | Profiler        | Started profiling model: Custom_model.tflite (Local Simulation)
2026-03-24 05:39:53  | Profiling        | Success  | Profiler        | Successfully profiled Custom_model.tflite - Arena: 91.7 KB, MACs: 8100000
----------------------------------------------------------------------------------------------------


#### Upload Logs
The logs files that are in json format can be uploaded and stored on to the databricks delta tables by specifying the table name and the warehouse_name. 

In [0]:
from silabs_mlops.logs import Logger
logger = Logger(
    warehouse_name="Serverless Starter Warehouse", 
    table_name="mlops_dev.default.logs" 
)
logger.sync_to_databricks()


--- Found 2 local logs. Syncing to mlops_dev.default.logs ---
Uploading... 2/2 completed.
✓ Successfully Bulk Synced 2 local logs into mlops_dev.default.logs!
